# Qwen3.5-4B WAF / NIDS classifier — rsLoRA fine-tune

Two parts. **Part A** trains, predicts and exports, then stops. **Restart the runtime**, then run
**Part B** in that fresh process: adapter reload, continuation test, and `gpu_smoke_test.json` — the
only evidence that counts for the *Notebook smoke-tested* state. Run `MODE="smoke"` first.

- Input: `colab_bundle_*.zip` (data + helpers + manifests) made by `training/make_bundle.py`.
- Training target: `{"threat_type": "<one of the taxonomy keys>"}` only. Loss on the answer + `<|im_end|>` only.
- 16-bit LoRA (rsLoRA r=16, alpha=32). QLoRA is deliberately not used (Unsloth: not recommended for Qwen3.5).
- Nothing is uploaded or published automatically. Package pins below are **candidates**: this exact set
  has not yet been run together; the smoke report records what actually got installed.

In [ ]:
# ---- configuration (WAF_* env vars override, e.g. `colab exec --env WAF_MODE=fast`) ------------
import os
MODE = os.environ.get("WAF_MODE", "smoke")   # "smoke" (>=20 optimizer steps) | "fast" (30k subset) | "full" (100k)
DRY_RUN_STEPS = int(os.environ.get("WAF_DRY_RUN_STEPS", 0)) or None
                          # e.g. 25 with MODE="fast": start-check on the real bundle (ckpt+eval every 10 steps,
                          # 60 predictions, Part B continuation test, full-run time estimate); None = real run
BUNDLE = f"/content/colab_bundle_{'pilot' if MODE == 'smoke' else MODE}.zip"   # upload via the Files pane, or Drive
DRIVE_ROOT = os.environ.get("WAF_DRIVE_ROOT", "/content/drive/MyDrive/qwen35_waf")  # "" = no Drive
                          # run dir = DRIVE_ROOT/<mode>[_dry]_<bundle sha8>: progress.json, train_log.jsonl,
                          # checkpoints (newest 2), Part A/B reports, adapter. Same bundle + mode resumes from it.
SAVE_EVERY_MIN = 20      # also checkpoint when this many minutes passed since the last save (disconnect loss cap)
BATCH = int(os.environ.get("WAF_BATCH", 0)) or None  # per-device batch; None = from GPU memory (T4 1, L4 2, A100-40 4, 80 GB 8), effective 16
RUN_RESUME_TEST = True    # smoke only: compare uninterrupted vs save/reload continuation
SEED = 3407
MODEL_ID = "unsloth/Qwen3.5-4B"
MODEL_REVISION = "3764fa359b9082ea5a1e4a5e3ac3aaf6e9671636"
MAX_LEN = 2048
EVAL_SUBSET = 1000        # interim evals use the first N val_select rows (a full 5k eval took ~56 min on a T4)
EVAL_EVERY = int(os.environ.get("WAF_EVAL_EVERY", 250))  # optimizer steps between eval + checkpoint (real runs)
GC = {"0": False, "1": True}.get(os.environ.get("WAF_GC", ""), "unsloth")  # gradient checkpointing (off: no recompute, more memory)
CCONV = os.environ.get("WAF_CCONV") == "1"  # build causal-conv1d (no torch-2.11 wheel upstream; sm80 source build)

In [ ]:
# ---- pinned stack (tested on Colab T4, torch 2.11.0+cu130, 2026-10-15). unsloth 2026.10.5 caps
# transformers<=5.17.0, trl<=1.15.0, datasets<5.0; torch pinned so xformers cannot swap the CUDA build
!pip install -q "torch==2.11.0" "unsloth==2026.10.5" "unsloth_zoo==2026.10.5" "transformers==5.17.0" "trl==1.15.0" "peft==0.21.2" "accelerate==1.15.0" "bitsandbytes==0.50.2" "datasets==4.8.5" "xformers==0.0.35" "torchao==0.18.0" pyyaml jsonschema
# Triton kernels for Qwen3.5's gated-delta-rule layers; without them transformers runs the PyTorch reference loop.
# fla-core only (no fla.models, which tracks other transformers versions). ponytail: sm80+ only, untested on T4.
import subprocess
if float(subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], capture_output=True, text=True).stdout.split()[0]) >= 8:
    !pip install -q "fla-core==0.5.2"
if CCONV:  # ponytail: sm80-only source build; a failure leaves the PyTorch fallback (recorded in ENV)
    !CAUSAL_CONV1D_FORCE_BUILD=TRUE TORCH_CUDA_ARCH_LIST="8.0" MAX_JOBS=8 timeout 1500 pip install -q --no-build-isolation "causal-conv1d==1.7.0"
# If pip reports a torch / CUDA conflict: Runtime -> Restart session, then run from the next cell.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.7/79.7 kB 3.8 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━ 13.3/30.0 MB 84.9 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━ 24.0/30.0 MB 49.7 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 30.0/30.0 MB 38.1 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.0/30.0 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/12.3 MB ? eta -:--:--

...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━ 3.0/3.4 MB 18.1 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/3.6 MB 13.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/230.5 kB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 9.9 MB/s eta 0:00:00


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.2/819.2 kB 17.7 MB/s eta 0:00:00


In [ ]:
import json, os, shutil, subprocess, sys, time, math, hashlib, zipfile, platform, random
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"
props = torch.cuda.get_device_properties(0)
BF16 = torch.cuda.is_bf16_supported() and props.major >= 8  # T4 (7.5) only emulates bf16 -> fp16
ENV = {"gpu": props.name, "gpu_mem_gb": round(props.total_memory / 2**30, 1),
       "capability": f"{props.major}.{props.minor}", "dtype": "bfloat16" if BF16 else "float16",
       "disk_free_gb": shutil.disk_usage("/content").free // 2**30, "python": platform.python_version(),
       "torch": torch.__version__, "cuda": torch.version.cuda}
for pkg in ["unsloth", "unsloth_zoo", "transformers", "trl", "peft", "accelerate", "bitsandbytes", "datasets", "fla"]:
    try:
        ENV[pkg] = __import__(pkg).__version__
    except Exception as e:
        ENV[pkg] = f"ERROR {e}"
print(json.dumps(ENV, indent=1))
if ENV["gpu_mem_gb"] < 14:
    print("WARNING: <14 GB VRAM. Unsloth cites ~10 GB for 4B bf16 LoRA; 2048-token batches may OOM.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!


{
 "gpu": "NVIDIA A100-SXM4-40GB",
 "gpu_mem_gb": 39.5,
 "capability": "8.0",
 "dtype": "bfloat16",
 "disk_free_gb": 192,
 "python": "3.13.15",
 "torch": "2.11.0+cu130",
 "cuda": "13.0",
 "unsloth": "2026.10.5",
 "unsloth_zoo": "2026.10.5",
 "transformers": "5.17.0",
 "trl": "1.15.0",
 "peft": "0.21.2",
 "accelerate": "1.15.0",
 "bitsandbytes": "0.50.2",
 "datasets": "4.8.5",
 "fla": "0.5.2"
}


In [ ]:
# ---- bundle: unpack + verify checksums, counts, gate ---------------------------------------
if (DRIVE_ROOT or BUNDLE.startswith("/content/drive")) and not os.path.ismount("/content/drive"):
    from google.colab import drive; drive.mount("/content/drive")  # CLI: `colab drivemount -s NAME` first (exec has no auth UI)
BUNDLE_SHA = hashlib.sha256(open(BUNDLE, "rb").read()).hexdigest()
DRIVE_DIR = f"{DRIVE_ROOT}/{MODE}{'_dry' if DRY_RUN_STEPS else ''}_{BUNDLE_SHA[:8]}" if DRIVE_ROOT else None
if DRIVE_DIR and DRY_RUN_STEPS:
    shutil.rmtree(DRIVE_DIR, ignore_errors=True)  # a dry run always starts fresh; real runs resume from here
if DRIVE_DIR:
    os.makedirs(DRIVE_DIR, exist_ok=True)
print("bundle sha256", BUNDLE_SHA[:12], "| Drive run dir:", DRIVE_DIR)
B = "/content/bundle"
shutil.rmtree(B, ignore_errors=True)
zipfile.ZipFile(BUNDLE).extractall(B)
sys.path.insert(0, B)
info = json.load(open(f"{B}/BUNDLE_INFO.json"))
manifest = json.load(open(f"{B}/datasets_ready/manifest.json"))
gate = json.load(open(f"{B}/reports/pre_colab_gate.json"))
assert gate["passed"] or MODE == "smoke", "pre-Colab gate failed; only smoke runs allowed on this bundle"
for f, meta in manifest["files"].items():
    p = f"{B}/datasets_ready/{f}"
    if os.path.exists(p):
        assert hashlib.sha256(open(p, "rb").read()).hexdigest() == meta["sha256"], f"checksum {f}"
        assert sum(1 for _ in open(p)) == meta["records"], f"count {f}"
from training import prepare as P
from eval.policy import parse_output
import yaml
TAX = yaml.safe_load(open(f"{B}/config/taxonomy.yaml"))["classes"]
FILES = {"smoke": "pilot_train.jsonl", "fast": "fast_train.jsonl", "full": "train_full_100k.jsonl"}
VAL = {"smoke": "pilot_val.jsonl", "fast": "fast_val.jsonl"}.get(MODE, "val_10k.jsonl")
print("bundle ok:", info["prefix"], "gate passed:", gate["passed"])

bundle sha256 055aea2e8966 | Drive run dir: None


bundle ok: fast_ gate passed: True


In [ ]:
# ---- model (16-bit LoRA, text-only path) -------------------------------------------------------
from unsloth import FastLanguageModel
from training import colab_helpers as H
model, processor = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID, revision=MODEL_REVISION, max_seq_length=MAX_LEN,
    load_in_4bit=False, load_in_16bit=True, full_finetuning=False)
tok = getattr(processor, "tokenizer", processor)   # text tokenizer even if a processor comes back
fp_now = {"chat_template_sha256": P.sha256(tok.chat_template),
          "system_prompt_sha256": P.sha256(P.system_prompt(P.Path(B)))}
fp_ref = gate["tokenizer_fingerprint"]
for k, v in fp_now.items():
    assert v == fp_ref[k], f"{k} mismatch: Colab {v} vs local gate {fp_ref[k]}"
assert tok.eos_token == "<|im_end|>", tok.eos_token
model = FastLanguageModel.get_peft_model(model, **{**H.LORA, "use_gradient_checkpointing": GC}, random_state=SEED, max_seq_length=MAX_LEN)
ENV["gradient_checkpointing"] = str(GC)
try:
    import causal_conv1d; ENV["causal_conv1d"] = causal_conv1d.__version__
except ImportError:
    ENV["causal_conv1d"] = None
trainable = [(n, p.numel()) for n, p in model.named_parameters() if p.requires_grad]
n_train, n_all = sum(n for _, n in trainable), sum(p.numel() for p in model.parameters())
assert not any("visual" in n or "vision" in n for n, _ in trainable), "vision params are trainable"
ENV["weights_dtype"] = str(next(p for n, p in model.named_parameters() if "lora" not in n).dtype)
print(f"trainable {n_train:,} / {n_all:,} ({100*n_train/n_all:.2f}%) in {len(trainable)} tensors | base weights {ENV['weights_dtype']}")
print("sample trainable modules:", sorted({n.split('.lora')[0].split('.')[-1] for n, _ in trainable}))
ENV["fla_kernels"] = "fla.ops.gated_delta_rule" in sys.modules  # False = PyTorch reference loop (slow)
print("fla gated-delta-rule kernels:", ENV["fla_kernels"])

==((====))==  Unsloth 2026.10.5: Fast Qwen3_5 patching. Transformers: 5.17.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


trainable 21,233,664 / 4,560,499,200 (0.47%) in 256 tensors | base weights torch.bfloat16
sample trainable modules: ['down_proj', 'gate_proj', 'k_proj', 'o_proj', 'q_proj', 'up_proj', 'v_proj']
fla gated-delta-rule kernels: True


In [ ]:
# ---- data: render with the checkpoint template, mask, verify ---------------------------------
train_recs = H.load_records(B, FILES[MODE])
val_recs = H.load_records(B, VAL, {"val_select"})          # model selection partition ONLY
train_rows, val_rows = H.encode(tok, train_recs), H.encode(tok, val_recs)
# mask inspection: supervised span must be exactly the target + <|im_end|>
for r, row in random.Random(SEED).sample(list(zip(train_recs, train_rows)), 5):
    sup = tok.decode([t for t, l in zip(row["input_ids"], row["labels"]) if l != P.IGNORE])
    assert sup == r["messages"][2]["content"] + "<|im_end|>", sup
    print("supervised:", repr(sup), "| prompt tokens masked:", sum(l == P.IGNORE for l in row["labels"]))
print(len(train_rows), "train /", len(val_rows), "val_select rows")

supervised: '{"threat_type":"jwt_auth_bypass"}<|im_end|>' | prompt tokens masked: 1432
supervised: '{"threat_type":"benign_standard_traffic"}<|im_end|>' | prompt tokens masked: 845
supervised: '{"threat_type":"web_cache_poisoning"}<|im_end|>' | prompt tokens masked: 968
supervised: '{"threat_type":"benign_hard_negative_search"}<|im_end|>' | prompt tokens masked: 612
supervised: '{"threat_type":"malicious_file_upload_webshell"}<|im_end|>' | prompt tokens masked: 1072
32391 train / 1934 val_select rows


In [ ]:
# ---- trainer ----------------------------------------------------------------------------------
from transformers import Trainer, TrainerCallback
from datasets import Dataset
collate = H.make_collate(tok.pad_token_id)

class Meter(TrainerCallback):
    """Token throughput over optimizer-step time only (evals and saves excluded, so short dry runs measure the same
    rate as real runs) after warmup steps; finite-loss check; progress + complete-checkpoint copies to Drive."""
    def __init__(self, tokens_per_step, warmup=2):
        self.tps, self.warm, self.t0, self.losses, self.tokens = tokens_per_step, warmup, None, [], 0
        self.last_save, self.busy, self.step_t = time.time(), 0.0, None
    def on_step_begin(self, args, state, control, **kw):
        self.step_t = time.time()
    def on_step_end(self, args, state, control, **kw):
        if state.global_step == self.warm: self.t0 = time.time(); self.tokens = 0; self.busy = 0.0
        elif state.global_step > self.warm and self.step_t: self.tokens += self.tps; self.busy += time.time() - self.step_t
        if time.time() - self.last_save > SAVE_EVERY_MIN * 60:
            control.should_save = True  # time-based checkpoint on top of save_steps
    def on_log(self, args, state, control, logs=None, **kw):
        if logs and "loss" in logs:
            assert math.isfinite(logs["loss"]), f"non-finite loss at step {state.global_step}"
            self.losses.append((state.global_step, logs["loss"]))
        if logs:  # heartbeat: survives a dead VM on Drive; `colab download` reads the /content copy
            tps = self.throughput()
            prog = {**logs, "step": state.global_step, "max_steps": state.max_steps, "tokens_per_s": tps,
                    "eta_h": (state.max_steps - state.global_step) * self.tps / tps / 3600 if tps else None,
                    "peak_mem_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2), "time": time.strftime("%F %T")}
            for d in filter(None, ["/content", DRIVE_DIR]):
                open(f"{d}/train_log.jsonl", "a").write(json.dumps(prog) + "\n")
                json.dump(prog, open(f"{d}/progress.json.tmp", "w")); os.replace(f"{d}/progress.json.tmp", f"{d}/progress.json")
    def on_save(self, args, state, control, **kw):
        ck = f"{args.output_dir}/checkpoint-{state.global_step}"
        self.last_save = time.time()
        open(f"{ck}/COMPLETE", "w").write(json.dumps({"step": state.global_step, "time": time.time()}))
        if DRIVE_DIR:
            os.makedirs(DRIVE_DIR, exist_ok=True)
            dst = f"{DRIVE_DIR}/checkpoint-{state.global_step}"
            shutil.rmtree(dst + ".part", ignore_errors=True); shutil.copytree(ck, dst + ".part")
            shutil.rmtree(dst, ignore_errors=True); os.replace(dst + ".part", dst)
            done = sorted((c for c in os.listdir(DRIVE_DIR) if c.startswith("checkpoint-") and not c.endswith(".part")),
                          key=lambda c: int(c.rsplit("-", 1)[1]))
            for old in done[:-2]:  # keep the newest 2 complete copies on Drive
                shutil.rmtree(f"{DRIVE_DIR}/{old}", ignore_errors=True)
    def throughput(self):
        return self.tokens / self.busy if self.t0 and self.busy else None

avg_tokens = sum(len(r["input_ids"]) for r in train_rows) / len(train_rows)
SHORT = MODE == "smoke" or bool(DRY_RUN_STEPS)
STEPS = 25 if MODE == "smoke" else (DRY_RUN_STEPS or -1)
OUT = f"/content/run_{MODE}" + ("_dry" if DRY_RUN_STEPS else "")
meter = Meter(tokens_per_step=avg_tokens * 16, warmup=min(20, STEPS // 5) if STEPS > 0 else 20)
args = H.make_args(OUT, STEPS, 10 if SHORT else EVAL_EVERY, smoke=SHORT, seed=SEED, per_device=BATCH)
print("batch", args.per_device_train_batch_size, "x accum", args.gradient_accumulation_steps, "| optim", args.optim.value, "| clip", args.max_grad_norm, "| warmup", args.warmup_steps, "| fp16", args.fp16, "| bf16", args.bf16)
trainer = Trainer(model=model, args=args, train_dataset=Dataset.from_list(train_rows),
                  eval_dataset=Dataset.from_list(val_rows[:100 if DRY_RUN_STEPS else EVAL_SUBSET]), data_collator=collate, callbacks=[meter])

batch 4 x accum 4 | optim adamw_8bit | clip 1.0 | warmup 0.05 | fp16 False | bf16 True


In [ ]:
# ---- train (resumes from the newest COMPLETE checkpoint locally or on Drive) -------------------
def latest_complete(*dirs):
    cks = [os.path.join(d, c) for d in dirs if d and os.path.isdir(d) for c in os.listdir(d)
           if c.startswith("checkpoint-") and not c.endswith(".part") and os.path.exists(os.path.join(d, c, "COMPLETE"))]
    return max(cks, key=lambda c: int(c.rsplit("-", 1)[1]), default=None)
resume = latest_complete(OUT, DRIVE_DIR)
print("resuming from", resume) if resume else print("fresh start")
if resume:  # the sampler and skip-ahead depend on the per-device batch: resume only with the batch the run started with
    started = json.load(open(f"{resume}/trainer_state.json")).get("train_batch_size")
    assert started in (None, args.per_device_train_batch_size),         f"checkpoint was trained with batch {started}; set BATCH = {started} (same GPU type) to resume"
torch.cuda.reset_peak_memory_stats()
t_start = time.time()
result = trainer.train(resume_from_checkpoint=resume)
TRAIN = {"global_steps": trainer.state.global_step, "train_loss": result.training_loss,
         "wall_s": round(time.time() - t_start, 1), "tokens_per_s_excl_warmup": meter.throughput(),
         "peak_mem_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2), "losses": meter.losses,
         "eval_losses": [(h["step"], h["eval_loss"]) for h in trainer.state.log_history if "eval_loss" in h],
         "resumed_from": resume}
assert TRAIN["global_steps"] >= 20 or MODE != "smoke", "smoke must complete >= 20 optimizer steps"
print(json.dumps({k: v for k, v in TRAIN.items() if k != "losses"}, indent=1))
if MODE != "smoke":  # in a dry run: the full-run estimate on this GPU
    n_steps = math.ceil(len(train_rows) / 16)
    print(f"estimate: {n_steps} steps; {n_steps * avg_tokens * 16 / TRAIN['tokens_per_s_excl_warmup'] / 3600:.1f} h at observed throughput")

fresh start


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 32,391 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 21,233,664 of 4,560,499,200 (0.47% trained)


`causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


OutOfMemoryError: CUDA out of memory. Tried to allocate 144.00 MiB. GPU 0 has a total capacity of 39.49 GiB of which 17.44 MiB is free. Including non-PyTorch memory, this process has 39.46 GiB memory in use. Of the allocated memory 37.50 GiB is allocated by PyTorch, and 1.44 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
# ---- predictions on val_select (diagnostic only; final test is a separate frozen protocol) ------
FastLanguageModel.for_inference(model)
preds = []
for r in val_recs[: 60 if SHORT else 500]:  # ponytail: 500 = +-2% diagnostic; all 1,934 cost ~30 A100-min, B1s scores the rest
    t = time.time()
    raw = H.generate_label(model, tok, r["messages"])
    preds.append({"id": r["id"], "gold": r["metadata"]["threat_type"], "raw": raw,
                  "pred": parse_output(raw, TAX), "latency_s": time.time() - t})
PRED = {"n": len(preds), "raw_json_valid_rate": sum(p["pred"] is not None for p in preds) / len(preds),
        "accuracy_diagnostic": sum(p["pred"] == p["gold"] for p in preds) / len(preds),
        "latency_p50_s": sorted(p["latency_s"] for p in preds)[len(preds) // 2]}
print(json.dumps(PRED, indent=1)); print([p["raw"] for p in preds[:5]])

OutOfMemoryError: CUDA out of memory. Tried to allocate 40.00 MiB. GPU 0 has a total capacity of 39.49 GiB of which 15.44 MiB is free. Including non-PyTorch memory, this process has 39.47 GiB memory in use. Of the allocated memory 37.50 GiB is allocated by PyTorch, and 1.44 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
# ---- export adapter; Part A ends here ---------------------------------------------------------
EXP = f"/content/export_{MODE}"
model.save_pretrained(EXP); tok.save_pretrained(EXP)
json.dump({"base_model": MODEL_ID, "revision": MODEL_REVISION, "fingerprint": fp_now, "env": ENV,
           "lora": {k: v for k, v in H.LORA.items() if k != "use_gradient_checkpointing"}},
          open(f"{EXP}/export_info.json", "w"), indent=1)
json.dump([{"messages": r["messages"][:2], "expect": p["raw"]} for r, p in zip(val_recs, preds[:5])],
          open("/content/reload_sample.json", "w"))
PART_A = {"mode": MODE, "dry_run_steps": DRY_RUN_STEPS, "seed": SEED, "model_id": MODEL_ID, "revision": MODEL_REVISION, "env": ENV,
          "fingerprint": fp_now, "trainable_params": n_train, "train": TRAIN, "predictions": PRED,
          "export_dir": EXP, "bundle_dir": B, "train_file": FILES[MODE], "bundle_sha256": BUNDLE_SHA, "drive_dir": DRIVE_DIR}
json.dump(PART_A, open("/content/part_a.json", "w"), indent=1, default=str)
if DRIVE_DIR:  # Part A results + adapter survive the VM
    for f in ("part_a.json", "reload_sample.json"):
        shutil.copy(f"/content/{f}", DRIVE_DIR)
    shutil.rmtree(f"{DRIVE_DIR}/export", ignore_errors=True); shutil.copytree(EXP, f"{DRIVE_DIR}/export")
print("Part A complete -> /content/part_a.json. Now: Runtime -> Restart session, then run Part B.")

Unsloth: Restored added_tokens_decoder metadata in /content/export_fast/tokenizer_config.json.


NameError: name 'TRAIN' is not defined